In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, f1_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
hrv_klasik = ["SDNN", "RMSSD", "pNN50"]
statistik_rr = ["median_rr", "cv_rr"]
engineered = ["log_SDNN", "log_RMSSD", "log_pNN50", "log_cv_rr",
              "RMSSD_SDNN_ratio", "pNN50_norm",
              "SDNN_delta", "RMSSD_delta", "pNN50_delta"]

def kategorikan_fitur(feat):
    if feat in hrv_klasik:
        return "HRV Klasik"
    elif feat in statistik_rr:
        return "Statistik RR"
    elif feat in engineered:
        return "Engineered"
    else:
        return "TSFEL"

feature_category = {f: kategorikan_fitur(f) for f in feature_cols}

color_map = {
    "HRV Klasik": "#2E4057",
    "Statistik RR": "#4C9F70",
    "Engineered": "#FFB84D",
    "TSFEL": "#6BCB77",
}

Hitung Permutation Importance per Class

In [ ]:
importance_per_class = {cls: {name: [] for name in model_names} for cls in le.classes_}

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
    print(f"── Fold {fold+1} ──")
    X_train_raw, y_train = X[train_idx], y[train_idx]
    X_test_raw,  y_test  = X[test_idx],  y[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_imp)
    X_test_sc  = scaler.transform(X_test_imp)

    for name, clf in models_baseline.items():
        clf.fit(X_train_sc, y_train)
        for class_idx, class_name in enumerate(le.classes_):
            def f1_for_class(y_true, y_pred, class_idx=class_idx):
                return f1_score(y_true, y_pred, labels=[class_idx], average="macro", zero_division=0)
            scorer = make_scorer(f1_for_class)
            perm = permutation_importance(
                clf, X_test_sc, y_test,
                n_repeats=10, random_state=42, scoring=scorer, n_jobs=-1
            )
            importance_per_class[class_name][name].append(perm.importances_mean)

importance_summary_per_class = {}
for class_name in le.classes_:
    importance_summary_per_class[class_name] = {}
    for name in model_names:
        imp_matrix = np.array(importance_per_class[class_name][name])
        df_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": imp_matrix.mean(axis=0),
            "importance_std": imp_matrix.std(axis=0),
        })
        importance_summary_per_class[class_name][name] = df_imp

print("Permutation importance per kelas selesai")

Hitung Permutation Importance per Dataset

In [ ]:
importance_per_dataset = {ds: {name: [] for name in model_names} for ds in np.unique(datasets)}

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
    print(f"── Fold {fold+1} ──")
    X_train_raw, y_train = X[train_idx], y[train_idx]
    X_test_raw,  y_test  = X[test_idx],  y[test_idx]
    test_datasets_fold   = datasets[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_imp)
    X_test_sc  = scaler.transform(X_test_imp)

    for name, clf in models_baseline.items():
        clf.fit(X_train_sc, y_train)
        for ds_name in np.unique(test_datasets_fold):
            mask = test_datasets_fold == ds_name
            if mask.sum() < 5:
                continue
            X_test_ds = X_test_sc[mask]
            y_test_ds = y_test[mask]

            perm = permutation_importance(
                clf, X_test_ds, y_test_ds,
                n_repeats=10, random_state=42, scoring="f1_macro", n_jobs=-1
            )
            importance_per_dataset[ds_name][name].append(perm.importances_mean)

importance_summary_per_dataset = {}
for ds_name in np.unique(datasets):
    importance_summary_per_dataset[ds_name] = {}
    for name in model_names:
        imp_list = importance_per_dataset[ds_name][name]
        if len(imp_list) == 0:
            importance_summary_per_dataset[ds_name][name] = None
            continue
        imp_matrix = np.array(imp_list)
        df_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": imp_matrix.mean(axis=0),
            "importance_std": imp_matrix.std(axis=0),
        })
        importance_summary_per_dataset[ds_name][name] = df_imp

print("Permutation importance per dataset selesai")

VISUALISATION PER CLASS


In [ ]:
for class_name in le.classes_:
    fig, axes = plt.subplots(2, 2, figsize=(24, 24))
    axes = axes.flatten()

    for i, name in enumerate(model_names):
        df_imp = importance_summary_per_class[class_name][name].copy()
        df_imp["CV"] = df_imp["importance_std"] / (df_imp["importance_mean"].abs() + 1e-8)
        df_imp["kategori"] = df_imp["feature"].map(feature_category)
        df_sorted = df_imp.sort_values("CV", ascending=True).reset_index(drop=True)

        ax = axes[i]
        y_pos = np.arange(len(df_sorted))
        colors = [color_map[k] for k in df_sorted["kategori"]]

        ax.barh(y_pos, df_sorted["CV"][::-1], color=colors[::-1], edgecolor="white")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_sorted["feature"][::-1], fontsize=10)
        ax.set_xlabel("Coefficient of Variation (CV)", fontsize=13)
        ax.set_title(name, fontweight="bold", fontsize=17)
        ax.tick_params(axis='x', labelsize=11)

    legend_elements = [mpatches.Patch(color=c, label=k) for k, c in color_map.items()]
    fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.015),
               ncol=4, fontsize=15, frameon=False)
    plt.suptitle(f"Konsistensi Seluruh Fitur untuk Mendeteksi: {class_name}",
                 fontsize=19, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(data_path, f"konsistensi_besar_{class_name}.png"), dpi=200, bbox_inches="tight")
    plt.show()

VISUALISATION PER DATASET

In [ ]:
for ds_name in np.unique(datasets):
    fig, axes = plt.subplots(2, 2, figsize=(24, 24))
    axes = axes.flatten()

    for i, name in enumerate(model_names):
        df_imp = importance_summary_per_dataset[ds_name][name]

        ax = axes[i]
        if df_imp is None:
            ax.text(0.5, 0.5, "Data tidak cukup\nuntuk dianalisis", ha="center", va="center", fontsize=14)
            ax.set_title(name, fontweight="bold", fontsize=17)
            continue

        df_imp = df_imp.copy()
        df_imp["kategori"] = df_imp["feature"].map(feature_category)
        df_sorted = df_imp.sort_values("importance_mean", ascending=False).reset_index(drop=True)

        y_pos = np.arange(len(df_sorted))
        colors = [color_map[k] for k in df_sorted["kategori"]]

        ax.barh(y_pos, df_sorted["importance_mean"][::-1],
                xerr=df_sorted["importance_std"][::-1],
                color=colors[::-1], edgecolor="white", capsize=2)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_sorted["feature"][::-1], fontsize=10)
        ax.set_xlabel("Kontribusi (Permutation Importance)", fontsize=13)
        ax.set_title(name, fontweight="bold", fontsize=17)
        ax.axvline(0, color="gray", linewidth=0.8)
        ax.tick_params(axis='x', labelsize=11)

    legend_elements = [mpatches.Patch(color=c, label=k) for k, c in color_map.items()]
    fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.015),
               ncol=4, fontsize=15, frameon=False)
    plt.suptitle(f"Kontribusi Seluruh Fitur pada Dataset: {ds_name}", #\n(Skenario Cross-Dataset)
                 fontsize=19, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(data_path, f"kontribusi_besar_{ds_name} fiks.png"), dpi=200, bbox_inches="tight")
    plt.show()